# ISE 427 Tıpta Yapay Zeka - Veri Ön İşleme (Preprocessing)

**Hazırlayan:** Yiğit Kalaycıoğlu
**Numara:** 

### Amaç ve Kapsam
Bu notebook, Keşifsel Veri Analizi (EDA) aşamasında elde edilen bulgular doğrultusunda ham veriyi makine öğrenmesi modellerine hazırlamak için gerekli dönüşümleri içerir.

### Uygulanan İşlemler
1.  **Veri Temizliği:** Eksik verilerin doldurulması ve aykırı değerlerin (outlier) baskılanması.
2.  **Özellik Mühendisliği:** Hedef değişkenin ikili (binary) formata çevrilmesi.
3.  **Veri Dönüşümü (Pipeline):**
    * *Sayısal Değişkenler:* Standartlaştırma (StandardScaler) ile ölçek farklarının giderilmesi.
    * *Kategorik Değişkenler:* One-Hot Encoding ile sayısal temsile dönüştürülmesi.
4.  **Veri Bölümleme:** Eğitim (%80) ve Test (%20) setlerinin oluşturulması.
5.  **Dışa Aktarım:** İşlenmiş verilerin ve pipeline nesnesinin sonraki aşama (Model Tuning) için kaydedilmesi.

---

In [1]:
# --- 1. Kütüphanelerin Yüklenmesi ---
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib  # Modelleri ve veriyi kaydetmek için
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


In [2]:
# --- 2. Veri Setinin Yüklenmesi ---
CSV_PATH = "Heart_disease_cleveland_new.csv"
try:
    df = pd.read_csv(CSV_PATH)
    print(f"Veri seti yüklendi. Boyut: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("Hata: CSV dosyası bulunamadı! Lütfen dosya yolunu kontrol edin.")

Veri seti yüklendi. Boyut: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,0,145,233,1,2,150,0,2.3,2,0,2,0
1,67,1,3,160,286,0,2,108,1,1.5,1,3,1,1
2,67,1,3,120,229,0,2,129,1,2.6,1,2,3,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0,1,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0,1,0


### 1: Veri Temizliği (Data Cleaning)
Bu aşamada eksik veriler (NaN) ve aykırı değerler (Outliers) ele alınacaktır.

* **Eksik Veriler:** Sayısal sütunlar medyan ile, kategorik sütunlar mod (en sık tekrar eden değer) ile doldurulur.
* **Aykırı Değerler:** EDA aşamasında tespit edilen uç değerler, IQR (Interquartile Range) yöntemi ile alt ve üst sınırlara baskılanır (Winsorization).

In [3]:
# 1.1. Eksik Değerlerin Doldurulması
missing_cols = df.columns[df.isnull().any()]
if len(missing_cols) > 0:
    print(f"Eksik değer içeren sütunlar dolduruluyor: {list(missing_cols)}")
    for c in df.columns:
        if df[c].dtype != 'object':
            df[c] = df[c].fillna(df[c].median())  # Sayısal -> Medyan
        else:
            df[c] = df[c].fillna(df[c].mode()[0]) # Kategorik -> Mod
else:
    print("Eksik değer bulunmamaktadır.")

# 1.2. Aykırı Değerlerin Baskılanması (IQR Yöntemi)
# Sadece sürekli sayısal değişkenlere uygulanır
numeric_outlier_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

print("\nAykırı değerler IQR sınırlarına baskılanıyor.")
for c in numeric_outlier_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Değerleri alt ve üst sınırlara çek (Clipping)
    df[c] = np.clip(df[c], lower_bound, upper_bound)

print("Veri temizliği tamamlandı.")

Eksik değer bulunmamaktadır.

Aykırı değerler IQR sınırlarına baskılanıyor.
Veri temizliği tamamlandı.


### 2: Değişkenlerin Ayrıştırılması ve Hedef Dönüşümü
Modelin doğru çalışması için değişken tiplerinin (sayısal vs kategorik) doğru tanımlanması ve hedef değişkenin uygun formata (0-1) getirilmesi gerekir.

In [4]:
# Değişken Listelerinin Tanımlanması
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
target = 'target'

# Hedef Değişken Dönüşümü
# Eğer hedef değişken çok sınıflıysa (0, 1, 2, 3, 4 gibi), bunu ikili (0: Sağlıklı, 1: Hasta) formata çeviriyoruz.
if df[target].nunique() > 2:
    print("Hedef değişken çok sınıflıdan ikili (binary) formata dönüştürülüyor.")
    df[target] = (df[target] > 0).astype(int)

# Özellikler (X) ve Hedef (y) ayrımı
X = df.drop(columns=[target])
y = df[target]

print(f"X Boyutu: {X.shape}, y Boyutu: {y.shape}")

X Boyutu: (303, 13), y Boyutu: (303,)


### 3: Eğitim ve Test Bölümlemesi (Train-Test Split)
Veri seti, modelin başarısını tarafsız ölçebilmek için **%80 Eğitim** ve **%20 Test** olarak ayrılır.

* **Stratify:** Hedef değişkenin (`y`) dağılım oranının (hasta/sağlıklı dengesi) hem eğitim hem test setinde aynı kalmasını sağlar.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Sınıf dengesini koru
)

print(f"Eğitim Seti: {X_train.shape[0]} örnek")
print(f"Test Seti  : {X_test.shape[0]} örnek")

Eğitim Seti: 242 örnek
Test Seti  : 61 örnek


### 4: Ön İşleme Pipeline'ının Kurulması
Veri sızıntısını (Data Leakage) önlemek için dönüşümler (Scaling, Encoding) sadece eğitim verisine göre öğrenilir (`fit`) ve test verisine uygulanır (`transform`).

1.  **Sayısal Değişkenler:** `StandardScaler` ile ortalaması 0, standart sapması 1 olacak şekilde ölçeklenir. (KNN ve SVM gibi mesafe temelli algoritmalar için kritiktir.)
2.  **Kategorik Değişkenler:** `OneHotEncoder` ile sayısal vektörlere dönüştürülür.

In [6]:
# Pipeline Tanımlama
preprocessing_pipeline = ColumnTransformer(
    transformers=[
        # Sayısal değişkenler için standardizasyon
        ('num', StandardScaler(), num_cols),
        # Kategorik değişkenler için One-Hot Encoding (bilinmeyen kategorileri yoksay)
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    verbose_feature_names_out=False # Sütun isimlerini temiz tut
)

# Pipeline'ı Eğitim Verisine Uygula (Fit & Transform)
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Test Verisini Sadece Dönüştür (Transform)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Ön işleme tamamlandı.")
print(f"İşlenmiş Eğitim Verisi Boyutu: {X_train_processed.shape}")
print(f"İşlenmiş Test Verisi Boyutu  : {X_test_processed.shape}")

Ön işleme tamamlandı.
İşlenmiş Eğitim Verisi Boyutu: (242, 28)
İşlenmiş Test Verisi Boyutu  : (61, 28)


### 5: Verilerin Kaydedilmesi
Hazırlanan veriler ve kurulan pipeline nesnesi, bir sonraki aşama olan **Model Eğitimi ve Hiperparametre Optimizasyonu** notebook'unda kullanılmak üzere diske kaydedilir.

In [7]:
# Dosya İsimleri
DATA_FILE = "train_test_data.pkl"
PIPELINE_FILE = "preprocess_pipeline.pkl"

# 1. İşlenmemiş (Ham) Eğitim/Test verilerini kaydet 
# (Not: Pipeline modelin içinde kullanılacağı için ham haliyle saklıyoruz)
joblib.dump((X_train, X_test, y_train, y_test), DATA_FILE)

# 2. Eğitilmiş Pipeline nesnesini kaydet
joblib.dump(preprocessing_pipeline, PIPELINE_FILE)

print(f"Veriler kaydedildi: {DATA_FILE}")
print(f"Pipeline kaydedildi: {PIPELINE_FILE}")
print("\nÖn işleme aşaması başarıyla tamamlandı! Bir sonraki notebook'a geçebilirsiniz.")

Veriler kaydedildi: train_test_data.pkl
Pipeline kaydedildi: preprocess_pipeline.pkl

Ön işleme aşaması başarıyla tamamlandı! Bir sonraki notebook'a geçebilirsiniz.
